In [ ]:
import sys
sys.path.append('../')  # FIX: moved above the repo imports below, which need it

import numpy as np
import torch
from kan import KAN

import stoich  # FIX: added, for the checked S and the harness version of this method


In [ ]:
from train import build_kan_width  # FIX: make_model dropped, cell below redefines it anyway


In [ ]:
# copied from train.py
def make_model(kan_width, seed, device):
    torch.manual_seed(seed)
    m = KAN(width=kan_width, grid=5, k=3, seed=seed, device=device, auto_save=False, grid_eps=0.0)
    m.speed()
    # FIX: speed() sets symbolic_enabled=False, and forward() then skips the symbolic branch entirely, so the
    # fixed last layer below would contribute nothing (its splines are masked to 0) and the model would output 0.
    m.symbolic_enabled = True
    return m


In [ ]:
width = build_kan_width(16)
width # nice

In [ ]:
model = make_model(width, seed=42, device='cpu')

In [ ]:
model # we got a model

In [ ]:
num_layers = model.depth
last_layer = model.depth - 1
print(last_layer) # cool 

model.act_fun[last_layer].in_dim, model.act_fun[last_layer].out_dim

# OK. we need to tweak our KAN constructor so that we get the in and out dim of the final layer to always be (num_rxns, num_species). In our case, 13 and 16

In [ ]:
def build_kan_width2(n_species, num_fluxes): # tweak your existing function
    return [n_species, 16, 16, num_fluxes, n_species]

In [ ]:
width = build_kan_width2(16, 13)
width # great

In [ ]:
# recreate a model
model = make_model(width, 42, 'cpu')
model

In [ ]:
last = model.depth-1
model.act_fun[last].in_dim, model.act_fun[last].out_dim # cool

# One idea (possibly too brute force)

We can create the stoichiometry matrix, and read off that to create our own functions. Below is the stoich matrix

In [ ]:
# stoichiometry matrix has dimensions (num_species, num_rxns)
# this could stand to be double checked
S = np.array([[1, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], # O3
              [1, -1, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0], # NO
              [-1, 1, 0, 0, 0, 1, -1, 0, 0, 0, 0, -1, 1], # NO2
              [0, 0, -1, -1, -1, 0, 0, 0, 0, 0, 0, 0, 0], # HCHO
              [0, 0, 2, 0, 1, -1, 0, 0, 1, 0, 1, 0, 0], # HO2
              [0, 0, 0, 0, 0, 0, 0, -1, -1, 0, 0, 0, 0], # H2O2
              [0, 0, 0, 0, -1, 1, -1, 2, -1, -1, 0, 0, 0], # OH
              [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], # HNO3
              [0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0], # CO
              [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], # H2
              [0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0], # ALD2
              [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0], # MGLY
              [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, -1, 1], # MCO3
              [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -1], # PAN
              [0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0], # H2O
              [-1, 1, -2, 0, -1, 0, 0, 0, 0, -1, -2, 0, 0]]) # O2  FIX: R5 (HCHO + OH + O2 -> HO2 + CO + H2O) was 0, is -1
S.shape


In [ ]:
# FIX: added check. every reaction must conserve C, H, N and O atoms, and S must match stoich.py's.
print("atoms off per reaction (C,H,N,O):", np.abs(stoich.atoms(stoich.SPECIES16) @ S).max(axis=1))
print("matches stoich.py:", np.array_equal(S, stoich.stoich_matrix(stoich.SPECIES16)))


## And here one way I figured out we can fix activations according to whatever function we want

The part that does it is the `lambda x: S_row[j_idx] * x` argument in `.fix_symbolic()`. 
The loop starting at Line 4 goes through 

1) Every node in the L-1 layer of the KAN (here, our 13 "reaction flux" nodes), 
2) Assigns scaled linear activation functions to each activation emanating from this reaction flux (16 of these, many of which are 0 as you can see from S above). This corresponds to the stoichiometry of each species participating in that reaction. Each activation points to a species tendency output node.

After this, activation function outputs are aggregated (summed) for each node. Since many activations are 0, this should be no problem.

In [ ]:
last_layer = model.act_fun[last]
LAST = model.depth-1

# loop through final act functions
for i_idx in range(last_layer.in_dim):
    S_row = S[:,i_idx]
    for j_idx in range(last_layer.out_dim):
        # FIX: the lambda looked S_row and j_idx up when called, not when made, so after the loop every edge used
        # S[15, 12] = 0. binding the coefficient as a default argument freezes it per edge.
        model.fix_symbolic(LAST, i_idx, j_idx, lambda x, c=float(S_row[j_idx]): c * x, fit_params_bool=False)

# freeze the whole layer
for p in model.act_fun[LAST].parameters():
    p.requires_grad_(False)
for p in model.symbolic_fun[LAST].parameters():
    p.requires_grad_(False)

print(model.act_fun[LAST].mask)                    # should be all 0 (splines off)
print(model.symbolic_fun[LAST].mask)               # should be all 1 (symbolic on)
print([p.requires_grad for p in model.act_fun[LAST].parameters()])


In [ ]:
# FIX: added check. the last layer has to compute S @ flux exactly; before the lambda fix this was all zeros.
flux = torch.randn(5, len(S[0]))
out, _ = model.symbolic_fun[LAST](flux)
print("last layer == S @ flux:", torch.allclose(out, flux @ torch.tensor(S, dtype=flux.dtype).T, atol=1e-6))


This above code chunk can exist in your `make_model()` function. You can also create a class that has this code chunk as a function, e.g. 

In [ ]:
class KANBuilder(torch.nn.Module):
    def __init__(self, kan_width, seed, device):
        super().__init__()

        self.kan_width = kan_width
        self.seed = seed
        self.device = device
        self.model = self.make_model()

    def make_model(self):
        torch.manual_seed(self.seed)
        m = KAN(width=self.kan_width, grid=5, k=3, seed=self.seed, device=self.device, auto_save=False, grid_eps=0.0)
        m.speed()
        m.symbolic_enabled = True  # FIX: see make_model above
        return m

    # FIX: was a placeholder signature `(self, ...)` with no body, which is a syntax error.
    def fix_activations(self, S):
        last = self.model.depth - 1
        for i_idx in range(S.shape[1]):
            for j_idx in range(S.shape[0]):
                self.model.fix_symbolic(last, i_idx, j_idx, lambda x, c=float(S[j_idx, i_idx]): c * x,
                                        fit_params_bool=False)
        for p in list(self.model.act_fun[last].parameters()) + list(self.model.symbolic_fun[last].parameters()):
            p.requires_grad_(False)


This might be overkill, and adds an annoying extra piece of code to manage, i.e.

In [ ]:
smog_kan = KANBuilder(build_kan_width2(16, 13), seed=42, device='cpu')  # FIX: constructor needs its arguments
smog_kan.fix_activations(S)
smog_kan.model


But you get the idea

## Caveats:

1) according to Claude (lol), updating grid will reset the requires_grad argument to True. SO, after every grid update, we'd have to reset the requires_grad for every function in the last layer. 
2) this could be fine especially if we prune prior to the first grid update. that is, all those 0 functions could be taken out, and an exhaustive loop to reset requires_grad/freeze functions wouldn't take as much compute (though negligible either way)

## Short-term solution (for beta-testing)
1) Can you try training one version of a model with fixed final layer activations with setting update_grid=False within `model.fit()`?

## Using this in training (added)

Two more things are needed before this layer can train against the harness's targets:

1. **Units.** The harness standardizes each species' tendency, `y = (D - mean) / scale`. Applying `S` directly to standardized outputs means atoms are conserved in standardized units, not in ppb, so conservation is lost once predictions are mapped back. The edge coefficient has to be `S[j, i] * flux_scale[i] / y_scale[j]` and the output bias `-y_mean[j] / y_scale[j]`, the same numbers `StoichHead` uses.
2. **Saving.** A lambda saves as `"anonymous"` in the checkpoint config, and `KAN.loadckpt` can't look that name up, so the checkpoint can't be reloaded. Using the library function `"x"` and putting the coefficient in the edge's affine `c` slot avoids that.

Both are in `stoich.fix_stoich_layer(model, head)`, which is this loop with those two changes. Run it from the harness with `stoich: symbolic` in a config.

On the caveats below: `update_grid` only rebuilds spline coefficients and leaves the symbolic layer and its frozen state alone. `refine` does build a fresh spline layer (trainable again, but still masked to 0, so it can't change the output); the harness re-freezes it after every refine.
